# Anomalous Sightings Archive 

## Imports & Variables 

In [ ]:
import pandas as pd
import pygeohash as pgh
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
import datetime
import sys
import os
import requests
import sqlite3
import time
import geopandas as gpd
from geodatasets import get_path

from io import StringIO
from datetime import datetime, time 
from pathlib import Path
from tqdm.auto import tqdm 
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from python.kp_index import update_kp_data
from python.geo_location import create_geohashes
from python.geo_location import haversine_distance
from python.weather_api import get_weather
from python.weather_api import classify_weather



## Data Cleaning & Enrichment for US UAP Reports 1940 - 2014 

### Read the UAP Dataset into a Pandas Dataframe
- "../data/raw/uap_original_dataset.csv"
- a few rows in this large dataset have an extra column of irrelevent data
    - the fix: usecols=range(0, 11)

In [ ]:
uap_df = pd.read_csv("../data/raw/uap_original_dataset.csv", usecols=range(0, 11), low_memory=False) 

uap_df.info()

In [ ]:
uap_df.head()

In [ ]:
# At least one row in the 'datetime' col contains an invalid 24:00 for the time
# AI Use: Grok assistance in fixing any entries with 24:00 in formating datetime

# Make a copy of the original column
col = uap_df['datetime'].astype(str)

# Handle 24:00 cases
mask_24 = col.str.contains('24:00', na=False)
col = col.str.replace('24:00', '00:00', regex=False)

# Convert to real datetime
uap_df['datetime'] = pd.to_datetime(col, format='%m/%d/%Y %H:%M', errors='coerce')

# Fix the date rollover for 24:00
uap_df.loc[mask_24, 'datetime'] = uap_df.loc[mask_24, 'datetime'] + pd.Timedelta(days=1)

# Create the columns 
uap_df['datetime_formatted'] = uap_df['datetime'].dt.strftime('%Y-%m-%d %H:%M')   # yyyy-mm-dd hh:mm
uap_df['full_date']           = uap_df['datetime'].dt.strftime('%Y-%m-%d')         # yyyy-mm-dd only

### Create Dataframe with Only US sightings After 1940

In [ ]:
us_uap_df = uap_df[uap_df["country"].str.lower() == "us"].copy()
us_uap_df.head()

In [ ]:
us_uap_after_1940_df = us_uap_df[
    us_uap_df["datetime"] >= '1940-01-01'
    
].copy()

us_uap_after_1940_df.info()

### Find out how many null values remain

In [ ]:
us_uap_after_1940_df.isnull().sum()

### Fix null values in 'shape' column
- begin with finding unique values 

In [ ]:
us_uap_after_1940_df['shape'].unique()

### Assign 'shape' NA and Null values as 'unknown'

In [ ]:
us_uap_after_1940_df['shape'] = us_uap_after_1940_df['shape'].fillna('unknown')
us_uap_after_1940_df.isnull().sum()

### fix durration (sec) null value - only one
- look at the row and decide to remove entry or fill in value .. 

In [ ]:
us_uap_after_1940_df[us_uap_after_1940_df['duration (seconds)'].isnull()]

# airport sighting, unclear durration based on other values - drop row in place 
us_uap_after_1940_df.dropna(subset=['duration (seconds)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

### remove 'duration hours / mins' 
- (null and wierd values + seconds column gives us duration and is more nomalized)

In [ ]:
us_uap_after_1940_df.drop(columns = ['duration (hours/min)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

### Fix null values in 'comments' column

In [ ]:
# what do the rows look like? 
us_uap_after_1940_df[us_uap_after_1940_df['comments'].isnull()]

# fill with null comments with 'no comment' 
us_uap_after_1940_df['comments'] = us_uap_after_1940_df['comments'].fillna('no comments')

us_uap_after_1940_df.isnull().sum()

### convert 'duration (seconds)' to int
- some entries have non-integer values in them
- use regex in conversion to remove non-integers 

In [ ]:
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(float)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].round(0)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(int)
us_uap_after_1940_df.head()

### Convert latitude to float type (longitude is already a float)

In [ ]:
us_uap_after_1940_df['latitude'] = us_uap_after_1940_df['latitude'].astype(float)

us_uap_after_1940_df.dtypes

### Convert 'full_date' to datetime + create 'year' and 'month' column

In [ ]:
us_uap_after_1940_df['full_date'] = pd.to_datetime(us_uap_after_1940_df['full_date'])

us_uap_after_1940_df['year'] = us_uap_after_1940_df['full_date'].astype(str).str[:4]
us_uap_after_1940_df['year'] = us_uap_after_1940_df['year'].astype(int)

us_uap_after_1940_df['month'] = us_uap_after_1940_df['full_date'].astype(str).str[5:7]
us_uap_after_1940_df['month'] = us_uap_after_1940_df['month'].astype(int)

us_uap_after_1940_df.head()
us_uap_after_1940_df.dtypes

### Enrich data even more with a 'season' column
- create season_groups dictionary
- map to 'month' value in each row

In [ ]:
season_groups = {
    '01': 'winter', 
    '02': 'winter', 
    '03': 'spring', 
    '04': 'spring', 
    '05': 'spring',
    '06': 'summer',
    '07': 'summer', 
    '08': 'summer',
    '09': 'fall', 
    '10': 'fall',
    '11': 'fall',
    '12': 'winter',
}

us_uap_after_1940_df['season'] = us_uap_after_1940_df['month'].astype(str).str.zfill(2).map(season_groups)

us_uap_after_1940_df.head()

### Add KP Index Data ('solar_kp_index', 'solar_ap_index' columns)
- Call update_kp_data function in module
- Read csv file into pandas df
- Ensure datetime is datetime 
- KP data is in UTZ and updated every 3 hours: for a given day, find the the data that matched to 9pm 21:00 US Central Time
- Create a 'full_date' column in the kp_9pm_us_cst_df to merge on.
- Merge to full_date to us uap sightings

In [ ]:
update_kp_data()

In [ ]:
kp_df = pd.read_csv("../data/processed/kp_index.csv", usecols=('datetime', 'kp', 'ap'))
kp_df.head(20)

In [ ]:
# ensure datetime column is a datetime type
kp_df['datetime'] = pd.to_datetime(kp_df['datetime'])
kp_df.dtypes

In [ ]:
# kp table includes values in utz (5-6 hours ahead of most US timezones)
# most uap sightings in the US happen late in the evening / close 9:00pm or 21:00 
# 03:00:00 or 3am UTZ would be the best match for most sightings .. which is 9pm central standard time 

# only 03:00 utz (it will have the next day's date, so we would need to convert to US time zome)
target_time = time(3, 0, 0)

# Filter for exactly 03:00 UTC
kp_3am_utz_df = kp_df[kp_df['datetime'].dt.time == target_time].copy()

# Convert to US Central Standard Time (CST = UTC-6)
kp_9pm_us_cst_df = kp_3am_utz_df.copy()
kp_9pm_us_cst_df['datetime'] = kp_9pm_us_cst_df['datetime'] - pd.Timedelta(hours=6)

kp_9pm_us_cst_df.head()



In [ ]:
# Create a 'full_date' column in the kp_9pm_us_cst_df to merge on

kp_9pm_us_cst_df['full_date'] = kp_9pm_us_cst_df['datetime'].dt.strftime('%Y-%m-%d')
kp_9pm_us_cst_df['full_date'] = pd.to_datetime(kp_9pm_us_cst_df['full_date'])
kp_9pm_us_cst_df.head()

In [ ]:
# Merge to full_date to us uap sightings

us_uap_after_1940_df = pd.merge(us_uap_after_1940_df, kp_9pm_us_cst_df[['full_date','kp', 'ap']], on='full_date', how='left')

us_uap_after_1940_df.head()

### Additional Cleanup
- create state_code column in uppercase to match bigfoot data and ERD
- rename columns to match ERD

In [ ]:
# create state_code column in uppercase to match bigfoot data and ERD

us_uap_after_1940_df['state_code'] = us_uap_after_1940_df['state'].str.upper()

# rename to match ERD

us_uap_after_1940_df = us_uap_after_1940_df.rename(
    columns={
        'duration (seconds)': 'duration_secs',
        'kp': 'solar_kp_index', 
        'ap': 'solar_ap_index'
    }
)

### Group Shapes - Enrich Data with 'shape_group' column
- There are many unique 'shapes' reported, but many are similar in 'type'
    - example: 'fireball' and 'flash' are both 'light' types
- Create a shape_groups dictionary
    - Missing or unknown values will be grouped in the 'other' category 
- Map to 'shape' column 

In [ ]:

# group shapes by types 

shape_groups = {
    'circle': 'round', 'sphere': 'round', 'disk': 'round', 
    'oval': 'round', 'round': 'round', 'crescent': 'round', 'dome': 'round',
    'egg': 'round', 'teardrop': 'round',
    
    'triangle': 'triangle', 'delta': 'triangle', 'chevron': 'triangle', 
    'pyramid': 'triangle', 'diamond': 'triangle',
    
    'cylinder': 'cigar', 'cigar': 'cigar',
    
    'light': 'light', 'fireball': 'light', 'flash': 'light', 'flare': 'light',
    
    'changing': 'changing', 'changed': 'changing', 'formation': 'changing', 'cone': 'changing',
    'cross': 'changing', 'hexagon': 'changing', 'rectangle': 'changing',
    
    'other': 'other', 
    'nan': 'other',
    'unknown': 'other',
    '': 'other'
}

us_uap_after_1940_df['shape_group'] = us_uap_after_1940_df['shape'].map(shape_groups)

us_uap_after_1940_df['shape_group'].value_counts()

### Check head (first 5 rows) and save to CSV in `data/processed/` folder 

In [ ]:
us_uap_after_1940_df.head()

In [ ]:
us_uap_after_1940_df.sort_values(by='datetime', ascending=True, inplace=True)

us_uap_after_1940_df['uap_id'] = range(1, len(us_uap_after_1940_df) +1)

In [ ]:
# create_geohashes function imported from python.geo_location
# create columns for geohash_7, 6, 5 ( use later for plotting, maps, and proximity )

us_uap_after_1940_df[['geohash_7', 'geohash_6', 'geohash_5']] = us_uap_after_1940_df.apply(
    lambda row: create_geohashes(row['latitude'], row['longitude']),
    axis=1,
    result_type='expand'
)

In [ ]:
# add rounded hour column
us_uap_after_1940_df['rounded_dt'] = us_uap_after_1940_df['datetime'].dt.round('h')
us_uap_after_1940_df['hour'] = us_uap_after_1940_df['rounded_dt'].dt.strftime('%H') 


In [ ]:
# reorder all columns to match ERD
# remove irrelevent columns - 'country' & 'date_posted'
# write to csv 

us_uap_after_1940_df = us_uap_after_1940_df[[
    'uap_id', 'datetime', 'city', 'state_code', 'state', 'duration_secs',
     'latitude', 'longitude', 'geohash_5', 'geohash_6', 'geohash_7',
     'full_date', 'year', 'month', 'season',  'datetime_formatted', 'rounded_dt', 'hour', 
     'shape', 'shape_group', 'comments',
     'solar_kp_index', 'solar_ap_index',        
    ]]


In [ ]:
us_uap_after_1940_df.columns

us_uap_after_1940_df.to_csv("../data/processed/us_uap_1940_v1.csv", index=False)

### Additional Enrichment Before Saving to Final

In [ ]:
us_uap_after_1940_df.dtypes

### Finished? Save to `../data/final/` folder 

In [ ]:
us_uap_after_1940_df.to_csv("../data/final/uap_reports.csv", index=False)

## Data Cleaning & Enrichment for Bigfoot Dataset(s)

### Read the first Bigfoot Dataset 
- '../data/raw/bfro_locations.csv'

In [ ]:
pd.set_option('display.max_columns', None) 

bigfoot_df = pd.read_csv('../data/raw/bfro_locations.csv', index_col=False)

bigfoot_df.shape
bigfoot_df.describe(include='all')
bigfoot_df.info()

In [ ]:
bigfoot_df.isna().sum()

### Rename 'number' to 'bf_id' 
- We will use this as the Primary Key later on
- Check datatypes 

In [ ]:
bigfoot_df = bigfoot_df.rename(columns={"number": "bf_id"})
bigfoot_df.dtypes

### Merge geohash data from the other bigfoot dataset 
- sample random entry to see if data matches
- rename 'number' to 'bf_id' in geohashed dataset
- merge on bf_id

In [ ]:
bigfoot_df.loc[bigfoot_df['bf_id'] == 799]

In [ ]:
# Read in geohashed dataset csv and ensure 799 is a match - test

bigfoot_geocoded_df = pd.read_csv('../data/raw/bfro_reports_geocoded.csv')
bigfoot_geocoded_df['number'] = bigfoot_geocoded_df['number'].astype(int)
bigfoot_geocoded_df.loc[bigfoot_geocoded_df['number'] == 799]

### Merge Bigfoot Dataframes 
- on 'bf_id' 
- new dataframe with everything we want → combined_bigfoot_df

In [ ]:
# so, it looks like we will need to do a left join from the bigfoot_df to the bigfoot_geocoded_df ... bf_id = number
# We need the state and city and wx data from the geocoded - so we will go through with the merge

bigfoot_geocoded_df = bigfoot_geocoded_df.rename(columns={"number": "bf_id"})
combined_bigfoot_df = pd.merge(
    bigfoot_df, 
    bigfoot_geocoded_df[[
        'bf_id', 
        'date', 
        'season', 
        'state', 
        'geohash',  
        'temperature_mid', 
        'precip_type', 
        'dew_point', 
        'cloud_cover', 
        'moon_phase', 
        'observed']], 
        on='bf_id', 
        how='left'
    )

combined_bigfoot_df.head()

### Check Datatypes 

In [ ]:
combined_bigfoot_df.dtypes

### Create Formatted 'full_date' Column 

In [ ]:
# full_date(datetime), year(int), month(int), dat(int)

combined_bigfoot_df['full_date'] = pd.to_datetime(combined_bigfoot_df['date'], format='%Y-%m-%d', errors='coerce')


combined_bigfoot_df.head()

In [ ]:
combined_bigfoot_df['full_date'].describe()

In [ ]:
combined_bigfoot_df = combined_bigfoot_df[combined_bigfoot_df['full_date'] > '1949']
combined_bigfoot_df['full_date'].describe()

#### Check Total NA and Missing Values

In [ ]:
combined_bigfoot_df.isna().sum()

### Note: dataset has many NA values - mostly missing weather data in older reports
    - solution : 2 tables 
    - reports and locations (main - with all bigfoot reports) and no weather data 
    - create seperate bigfoot_weather_df - weather data for reports with weather data 
    - bf_id is the unique identifier and primary key in both 

### Create the 'bigfoot_weather_df'
- only bigfoot reports with weather data
- include bf_id, weather data columns, and a few other columns
- leave out the other columns (the main dataframe with all sightings will contain them)

In [ ]:
bigfoot_weather_df = combined_bigfoot_df.copy()

# drop all NA values (mostly missing weather data)
bigfoot_weather_df.dropna(inplace=True)

bigfoot_weather_df.isna().sum()



In [ ]:
# include only weather related and essential columns (like bf_id) in bigfoot_weather_df
# leave out 'precip_type' - not accurate

bigfoot_weather_df = bigfoot_weather_df[[
    'bf_id', 'full_date', 'latitude', 'longitude', 'season', 
    'temperature_mid', 'dew_point', 'cloud_cover', 'moon_phase'
]]

# verify

bigfoot_weather_df.dtypes 

In [ ]:
# round the F temp down to 1 decimal point
bigfoot_weather_df['temperature_mid'] = round(bigfoot_weather_df['temperature_mid'], 1)

# raname to temperature_f for consistency
bigfoot_weather_df.rename(columns={"temperature_mid" : "temperature_f"}, inplace=True)

# round down to 1 decimal 
bigfoot_weather_df['dew_point'] = round(bigfoot_weather_df['dew_point'], 1)

# sort by bf_id
bigfoot_weather_df = bigfoot_weather_df.drop_duplicates(subset=['bf_id'])

bigfoot_weather_df = bigfoot_weather_df.sort_values(by='bf_id', ascending=True)

# check head
bigfoot_weather_df.head()

In [ ]:
# cloud_cover * 100 to normalize and match UAP cloud cover

bigfoot_weather_df['cloud_cover'] = bigfoot_weather_df['cloud_cover'] * 100 

In [ ]:
# write to csv in processed folder

bigfoot_weather_df.to_csv("../data/processed/bigfoot_weather_processed.csv", index=False)

In [ ]:
# create bigfoot_weather_final_df for final enrichment

bigfoot_weather_final_df = pd.read_csv("../data/processed/bigfoot_weather_processed.csv")

In [ ]:
bigfoot_weather_df.isna().sum()

In [ ]:
bigfoot_weather_final_df.dtypes

In [ ]:
# 'season' column contains Unknown values 

bigfoot_weather_final_df['season'].unique()

In [ ]:
bigfoot_weather_final_df['full_date'] = pd.to_datetime(bigfoot_weather_final_df['full_date'], format='%Y-%m-%d', errors='coerce')
bigfoot_weather_final_df['month'] = bigfoot_weather_final_df['full_date'].dt.month.astype(int)

season_groups = {
    '01': 'winter', 
    '02': 'winter', 
    '03': 'spring', 
    '04': 'spring', 
    '05': 'spring',
    '06': 'summer',
    '07': 'summer', 
    '08': 'summer',
    '09': 'fall', 
    '10': 'fall',
    '11': 'fall',
    '12': 'winter',
}

bigfoot_weather_final_df['season'] = bigfoot_weather_final_df['month'].astype(str).str.zfill(2).map(season_groups)

bigfoot_weather_final_df['season'].unique()


In [ ]:
bigfoot_weather_final_df = bigfoot_weather_final_df[[
    'bf_id', 'full_date', 'month', 'latitude', 'longitude', 
    'season', 'temperature_f', 'dew_point', 'cloud_cover', 'moon_phase']]

bigfoot_weather_final_df.info()

In [ ]:
# write to csv in final folder

bigfoot_weather_final_df.to_csv("../data/final/bigfoot_weather.csv", index=False)

### Clean Up The Main 'combined_bigfoot_df' by dropping the weather columns 
- drop weather columns from the main dataframe with all sightings
- all weather data is in the bigfoot_weather_df we created earlier

In [ ]:
# drop weather columns from combined_bigfoot ( since we have a bigfoot weather table )

combined_bigfoot_df.drop(columns=['precip_type', 'temperature_mid', 'dew_point', 'cloud_cover', 'moon_phase'], inplace=True)
combined_bigfoot_df.isna().sum()

In [ ]:
combined_bigfoot_df.dtypes

In [ ]:
combined_bigfoot_df.head()

In [ ]:
# fill in NA values in observed with the title (the closest to anecdotal / descriptive info)
combined_bigfoot_df['observed'] = combined_bigfoot_df['observed'].fillna(combined_bigfoot_df['title'])

In [ ]:
combined_bigfoot_df.isna().sum()

In [ ]:
combined_bigfoot_df.head()

### Merge KP Index data - use kp_9pm_us_cst_df['full_date']

> Note: Kp Data enrichment on the main dataframe (not 'bigfoot_weather_df')
> Kp data goes back to 1930s, so we can include it in the main 'combined_bigfoot_df'

In [ ]:
# left merge kp data for the date of each sighting 
combined_bigfoot_df = pd.merge(combined_bigfoot_df, kp_9pm_us_cst_df[['full_date','kp', 'ap']], on='full_date', how='left')

combined_bigfoot_df.head()

In [ ]:
# bigfoot geohash columns
 
combined_bigfoot_df['geohash_5'] = combined_bigfoot_df['geohash'].str[:5]
combined_bigfoot_df['geohash_6'] = combined_bigfoot_df['geohash'].str[:6]
combined_bigfoot_df['geohash_7'] = combined_bigfoot_df['geohash'].str[:7]

# check the head

combined_bigfoot_df.head()

### Mapping dictionary to create state_code column (unique id for state)
- this dictionary will be reused to nomalize the state_code column in other datasets 
- AI (grok4) used to generate dictionary (see AI Assistance section in readme)

In [ ]:
state_to_code = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC'
}

combined_bigfoot_df['state_code'] = combined_bigfoot_df['state'].map(state_to_code)
combined_bigfoot_df['state_code'] = combined_bigfoot_df['state_code'].fillna('Unknown')

combined_bigfoot_df['year']  = combined_bigfoot_df['full_date'].dt.year.astype(int)
combined_bigfoot_df['month'] = combined_bigfoot_df['full_date'].dt.month.astype(int)
combined_bigfoot_df['day']   = combined_bigfoot_df['full_date'].dt.day.astype(int)


combined_bigfoot_df.head()

In [ ]:
# fix unknown values in season group and normalize with lowercase

season_groups = {
    '01': 'winter', 
    '02': 'winter', 
    '03': 'spring', 
    '04': 'spring', 
    '05': 'spring',
    '06': 'summer',
    '07': 'summer', 
    '08': 'summer',
    '09': 'fall', 
    '10': 'fall',
    '11': 'fall',
    '12': 'winter',
}

combined_bigfoot_df['season'] = combined_bigfoot_df['month'].astype(str).str.zfill(2).map(season_groups)

combined_bigfoot_df['season'].unique()

In [ ]:
combined_bigfoot_df['full_date'].describe()

### check datatypes

In [ ]:
combined_bigfoot_df.dtypes

### reorder columns, drop duplicate rows and write to csv in '../data/processed/'

In [ ]:
combined_bigfoot_df = combined_bigfoot_df[[
    'bf_id', 'full_date', 'title', 'state_code', 'state', 
    'latitude', 'longitude', 'geohash_5', 'geohash_6', 'geohash_7', 'geohash', 
    'date', 'year', 'month', 'day', 'season',
    'classification',  'observed',
    'kp', 'ap'
]]

combined_bigfoot_df = combined_bigfoot_df.drop_duplicates(subset=['bf_id'])

combined_bigfoot_df.sort_values(by='bf_id', ascending=True, inplace=True)

combined_bigfoot_df = combined_bigfoot_df.rename(columns={
    'kp': 'solar_kp_index',
    'ap': 'solar_ap_index'
})

combined_bigfoot_df.to_csv("../data/processed/combined_bigfoot_v1.csv", index=False)

### save final version of bigfoot dataset to '../data/final/'

In [ ]:
combined_bigfoot_df.to_csv("../data/final/bigfoot_reports.csv", index=False)

## States Dataset

In [ ]:
# Read original CSV 

states_df = pd.read_csv("../data/raw/US-population-by-state(wide).csv")

# Filter for 2010 and transpose
states_2010 = states_df[states_df['year'] == 2010].copy()

# Transpose so states become rows
states_2010_pop_df = states_2010.transpose().reset_index()

states_2010_pop_df.head(10)

In [ ]:
# Drop the unnecessary 'year' row that got transposed
states_2010_pop_df = states_2010_pop_df.drop(0).reset_index(drop=True)

# Rename columns 
states_2010_pop_df.columns = ['state', '2010_population']

# Drop any completely empty rows 
states_2010_pop_df = states_2010_pop_df.dropna(how='all')

In [ ]:
# Use state_to_code dictionary to map state codes

states_2010_pop_df['state_code'] = states_2010_pop_df['state'].map(state_to_code)
states_2010_pop_df['state_code'] = states_2010_pop_df['state_code'].fillna('Unknown')

states_2010_pop_df.head()


In [ ]:
# Move state_code to first column - since it is the primary key
states_2010_pop_df.insert(0, 'state_code', states_2010_pop_df.pop('state_code'))

# Order By State Code 

states_2010_pop_df = states_2010_pop_df.sort_values(by='state_code')

# Drop NA Values (currently 1 empty row at the end)
states_2010_pop_df.dropna()

# Make 2010_population an int
states_2010_pop_df['2010_population'] = states_2010_pop_df['2010_population'].round().astype(int)

# Check that all 50 States + DC are included 

states_2010_pop_df.head(51)

### Add in state_area data to get population density
- `../data/raw/state_area.csv` created from AI generated 2 Column CSV using Census DATA:
    - `https://www.census.gov/geographies/reference-files/2010/geo/state-area.html`
- 'state_area' column to 'states_2010_pop_df'
- Enrich with 'population_density' column (population / area)

In [ ]:
# read the state_area.csv into dataframe

states_area_df = pd.read_csv("../data/raw/state_area.csv")


# merge states_pop_df with states_area_df on 'state_code'

states_complete_df = pd.merge(
    states_2010_pop_df, 
    states_area_df[[
        'state_code', 
        'land_area_sq_miles']], 
        on='state_code', 
        how='left'
    )

states_complete_df['pop_density'] = (
    states_complete_df['2010_population'] / states_complete_df['land_area_sq_miles']
).round().astype(int)

states_complete_df.head()


### Write to ../data/processed/states_processed_data.csv

In [ ]:
# Write to ../data/processed/

states_complete_df.to_csv("../data/processed/states_processed_data.csv", index=False)

### Write to ../data/final/states.csv

In [ ]:
# Write to ../data/final/

states_complete_df.to_csv("../data/final/states.csv", index=False)

## Proximity Table
- Bigfoot reports with any UAP in close proximity
    - distance: must at least match on geohash_6 (most important)

### Merge the bigfoot_df and uap_df on 'geohash_6'

In [ ]:
bigfoot_df = combined_bigfoot_df.copy()
uap_df = us_uap_after_1940_df.copy()

proximity_df = pd.merge(
    bigfoot_df[[
        'bf_id', 'latitude', 'longitude', 'full_date',
        'geohash_7', 'geohash_6', 'geohash_5', 'year'
    ]],
    uap_df[[
        'uap_id', 'full_date', 'city', 'state_code',
        'shape_group', 'geohash_7', 'geohash_6', 'geohash_5', 'latitude', 'longitude', 'year'
    ]],
    on='geohash_6',
    how='inner',
    suffixes=('_bf', '_uap')
)

proximity_df.head()

### Enrich with date_diff_days column
- this will be used in calcualting the proximity score

In [ ]:
# date difference in days

proximity_df['full_date_bf'] = pd.to_datetime(proximity_df['full_date_bf'], format="%Y-%m-%d", errors="coerce")
proximity_df['full_date_uap'] = pd.to_datetime(proximity_df['full_date_uap'], format="%Y-%m-%d", errors="coerce")

proximity_df['date_diff_days'] = (
    proximity_df['full_date_bf'] - proximity_df['full_date_uap']
).dt.days.abs()

### Order the Columns 

In [ ]:
proximity_df = proximity_df[[
    'bf_id', 'uap_id',
    'full_date_bf', 'full_date_uap', 'date_diff_days',
    'geohash_6', 'geohash_7_bf', 'geohash_7_uap',
    'city', 'state_code',
    'year_bf', 'year_uap',
    'latitude_bf', 'longitude_bf', 'latitude_uap', 'longitude_uap'
]]

proximity_df.head()

### Enrich with an exact 'distance_meters' column
- use haversine_distance() function 
- this will also be used to calculate the proximity score later on

In [ ]:
proximity_df['distance_meters'] = round (1000 * haversine_distance(
    proximity_df['latitude_bf'],
    proximity_df['longitude_bf'],
    proximity_df['latitude_uap'],
    proximity_df['longitude_uap']
))

# verify (descending, since many values are 0)
proximity_df.sort_values('distance_meters', ascending=False).head(20)

### Proximity Score
- Enrich with 'proximity_score column' to measure the strength of the match
- This factors in distance between the reports and time difference in days

In [ ]:
# proximity_score 
# distance and time decay 
# avoids divide by zero error
# generates an appoximate range of possible values 0 - 100 

proximity_df['proximity_score'] = round(
    100 * (
        1 / (1 + proximity_df['distance_meters'] / 5000) *      
        1 / (1 + proximity_df['date_diff_days'] / 365)         
    ), 2)

proximity_df['proximity_score'].describe()

### Proximity Rank

In [ ]:
proximity_df['proximity_rank'] = proximity_df['proximity_score'].rank(method='dense', ascending=False)

proximity_df['proximity_rank'].describe()

In [ ]:
proximity_df = proximity_df.sort_values('proximity_rank', ascending=True)

proximity_df.head()

### Re-order Columns to Match ERD

In [ ]:
# Re-order columns
# IDs, Location, Time, Proximity 
# Sort by Proximity Rank 

ordered_cols = [
    'bf_id', 'uap_id',

    'city', 'state_code',
    'geohash_6', 'geohash_7_bf', 'geohash_7_uap',
    'latitude_bf', 'longitude_bf',
    'latitude_uap', 'longitude_uap',
    
    'full_date_bf', 'full_date_uap', 'date_diff_days',
    'year_bf', 'year_uap',
    
    'distance_meters',
    'proximity_score',
    'proximity_rank'          
]

proximity_df = proximity_df[ordered_cols].sort_values(
    by='proximity_rank', 
    ascending=True 
).reset_index(drop=True)

### check for null values

In [ ]:
proximity_df.isnull().sum()

### write to ../data/processed/

In [ ]:
proximity_df.to_csv("../data/processed/proximity_v1.csv", index=False)

### write to ../data/final/

In [ ]:
proximity_df.to_csv("../data/final/proximity.csv", index=False)

## UAP Weather Data
- Due to the sheer number of reports and limits of the free API, we will take a sampling of Year 2011 reports

In [ ]:
us_uap_after_1940_df.dtypes

In [ ]:
# use .copy in previous kernels 
# create dataframe with just the 2011 sightings
# sample of approx 6000 sightings
# use open meteo api (no api key required) 
# create columns for weather data
# recreate sql db

us_uap_2011_wx_df = us_uap_after_1940_df.loc[
    us_uap_after_1940_df['year'] == 2011,
    ['uap_id', 'datetime', 'city', 'state_code', 
     'latitude', 'longitude', 'geohash_7',
     'year', 'month', 'season', 'full_date', 'rounded_dt', 'hour',
     'solar_kp_index', 'solar_ap_index']
].copy()

us_uap_2011_wx_df.shape

### Add weather columns before running the weather fetch loop

In [ ]:
us_uap_2011_wx_df['full_date'] = pd.to_datetime(us_uap_2011_wx_df['full_date'])
us_uap_2011_wx_df['full_date_str'] = us_uap_2011_wx_df['full_date'].dt.strftime('%Y-%m-%d')

# Add weather columns if missing
weather_cols = ["temperature_f", "dew_point", "cloud_cover", "precip_in", "wind_mph", "weather_conditions"]
for col in weather_cols:
    if col not in us_uap_2011_wx_df.columns:
        us_uap_2011_wx_df[col] = None

print(f"Loaded {len(us_uap_2011_wx_df):,} records")

### Weather Fetch Loop

- uses get_weather() function 
- checkpoints to save progress 

In [ ]:

CHECKPOINT = "../data/processed/us_uap_2011_wx_checkpoint.csv"

# clean file - ready for final weather condition enrichment
FINAL_OUT  = "../data/processed/us_uap_2011_wx.csv"   

# 1. Resume from the latest checkpoint
if os.path.exists(CHECKPOINT):
    us_uap_2011_wx_df = pd.read_csv(CHECKPOINT)
    print(f"Loaded checkpoint → {len(us_uap_2011_wx_df):,} rows")
else:
    print("No checkpoint found – starting from the original DataFrame")

# check number of missing rows
missing = us_uap_2011_wx_df["temperature_f"].isna().sum()
print(f"Rows still missing weather data: {missing:,}")

if missing == 0:
    print("Everything already filled – nothing to do.")
else:
    # Resume loop - skips any row that already has temperature_f
    for idx, row in tqdm(us_uap_2011_wx_df.iterrows(), total=len(us_uap_2011_wx_df)):
        if pd.notna(row["temperature_f"]):
            continue

        try:
            weather = get_weather(
                lat=row["latitude"],
                lon=row["longitude"],
                date_str=row["full_date"],
                hour=row["hour"]
            )
            for key, value in weather.items():
                us_uap_2011_wx_df.at[idx, key] = value

        except Exception as e:
            print(f"Error at row {idx}: {e}")
            # Optional: back off on rate-limit style errors
            if "limit" in str(e).lower() or "429" in str(e):
                print("Rate-limit suspected – sleeping 60 s …")
                time.sleep(60)
            # the next run will retry this row

        # Checkpoint 
        if (idx + 1) % 200 == 0:          
            us_uap_2011_wx_df.to_csv(CHECKPOINT, index=False)
            print(f"Checkpoint saved at row {idx + 1:,}")

    # Final save
    us_uap_2011_wx_df.to_csv(CHECKPOINT, index=False)
    us_uap_2011_wx_df.to_csv(FINAL_OUT, index=False)
    print("Finished – final files written.")

### Enrich 'weather_conditions' column - based on weather api data
- uses classify_weather() function
- save enriched dataframe to CSV in ../data/processed/ folder

In [ ]:
us_uap_2011_wx_df = pd.read_csv("../data/processed/us_uap_2011_wx.csv")
us_uap_2011_wx_df = classify_weather(us_uap_2011_wx_df)
print("\nWeather conditions distribution:")
us_uap_2011_wx_df['weather_conditions'].value_counts()

### Write to '../data/final/' Folder
- before doing this, remove columns that are no longer needed

In [ ]:
# write to final/ folder and omit extra date columns (not needed after weather enrichment)

us_uap_2011_wx_final_df = us_uap_2011_wx_df[[
    'uap_id', 'datetime', 'city', 'state_code', 'latitude', 'longitude', 'geohash_7',
    'year', 'month', 'season', 'hour',
    'solar_kp_index', 'solar_ap_index',
    'temperature_f', 'dew_point', 'cloud_cover', 'precip_in', 
    'wind_mph', 'weather_conditions'
]]

final_path = "../data/final/us_uap_2011_weather.csv"
us_uap_2011_wx_final_df.to_csv(final_path, index=False)

print(f"\nFile saved to: {final_path}")
print(f"Total rows: {len(us_uap_2011_wx_final_df):,}")
print(f"Columns saved: {len(us_uap_2011_wx_final_df.columns)}")


## Relational Database Creation & Queries - SQLite 

### Database Creation ( ../data/sql/anomalous_sightings.db )

In [ ]:
# Database Path and Connection
db_path = Path("../data/sql/")

# create the folder if missing due to gitignore
db_path.mkdir(parents=True, exist_ok=True)   
connection = sqlite3.connect(db_path / "anomalous_sightings.db")

# US UAP Reports Table (1940 - 2014)
uap_df = pd.read_csv("../data/final/uap_reports.csv")

uap_df.to_sql(
    name='uap_reports', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'uap_id': 'INTEGER',
        'datetime': 'TEXT',
        'city': 'TEXT',
        'state_code': 'TEXT',
        'state': 'TEXT',
        'duration_secs': 'INTEGER',
        'latitude': 'REAL',
        'longitude': 'REAL',
        'geohash_5': 'TEXT',
        'geohash_6': 'TEXT',
        'geohash_7': 'TEXT',
        'full_date': 'TEXT',
        'year': 'INTEGER',
        'month': 'INTEGER',
        'season': 'TEXT',
        'datetime_formatted': 'TEXT',
        'shape': 'TEXT',
        'shape_group': 'TEXT',
        'comments': 'TEXT',
        'solar_kp_index': 'REAL',
        'solar_ap_index': 'REAL'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_uap_id ON uap_reports(uap_id)")

# Bigfoot Weather Table 
bigfoot_wx_df = pd.read_csv("../data/final/bigfoot_weather.csv") 

bigfoot_wx_df.to_sql(
        name='bigfoot_weather', 
        con=connection, 
        if_exists='replace', 
        index=False,
        dtype={
            'bf_id': 'INTEGER',
            'full_date': 'TEXT',
            'month': 'INTEGER',
            'latitude': 'REAL', 
            'longitude': 'REAL',
            'season': 'TEXT', 
            'temperature_f': 'REAL', 
            'dew_point': 'REAL', 
            'cloud_cover': 'REAL', 
            'moon_phase': 'REAL'
        }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_bf_id ON bigfoot_weather(bf_id)")


# US Bigfoot Reports Table (2015 - 2021)
bigfoot_df = pd.read_csv("../data/final/bigfoot_reports.csv")

bigfoot_df.to_sql(
    name='bigfoot_reports', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'bf_id': 'INTEGER',
        'full_date': 'TEXT', 
        'title': 'TEXT', 
        'state_code': 'TEXT', 
        'state': 'TEXT', 
        'latitude': 'REAL', 
        'longitude': 'REAL', 
        'geohash_5': 'TEXT', 
        'geohash_6': 'TEXT', 
        'geohash_7': 'TEXT', 
        'geohash': 'TEXT', 
        'date': 'TEXT', 
        'year': 'INTEGER', 
        'month': 'INTEGER', 
        'day': 'INTEGER', 
        'season': 'TEXT',  
        'classification': 'TEXT', 
        'observed': 'TEXT',
        'solar_kp_index': 'REAL',
        'solar_ap_index': 'REAL'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_bf_id ON bigfoot_reports(bf_id)")



# States Table
states_df = pd.read_csv("../data/final/states.csv")

states_df.to_sql(
    name='states', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'state_code': 'TEXT',
        'state': 'TEXT', 
        '2010_population': 'INTEGER',
        'land_area_sq_miles': 'INTEGER',
        'pop_density': 'INTEGER'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_state_code ON states(state_code)")


# Proximity Table
proximity_df = pd.read_csv("../data/final/proximity.csv")

proximity_df.to_sql(
    name='proximity', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'bf_id': 'INTEGER', 
        'uap_id': 'INTEGER', 
        'city': 'TEXT', 
        'state_code': 'TEXT', 
        'geohash_6': 'TEXT', 
        'geohash_7_bf': 'TEXT', 
        'geohash_7_uap': 'TEXT', 
        'latitude_bf': 'REAL', 
        'longitude_bf': 'REAL', 
        'latitude_uap': 'REAL', 
        'longitude_uap': 'REAL', 
        'full_date_bf': 'TEXT', 
        'full_date_uap': 'TEXT', 
        'date_diff_days': 'INTEGER',
        'year_bf': 'INTEGER', 
        'year_uap': 'INTEGER', 
        'distance_meters': 'REAL', 
        'proximity_score': 'REAL',
        'proximity_rank': 'INTEGER'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_proximity_pk ON proximity(bf_id, uap_id)")

# US UAP 2011 Reports With Weather Enrichment
us_uap_2011_weather = pd.read_csv("../data/final/us_uap_2011_weather.csv")

us_uap_2011_weather.to_sql(
    name='us_uap_2011_weather', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'uap_id': 'INTEGER',
        'datetime': 'TEXT', 
        'city': 'TEXT', 
        'state_code': 'TEXT',   
        'latitude': 'REAL', 
        'longitude': 'REAL',
        'geohash_7': 'TEXT',
        'year': 'INTEGER',
        'month': 'INTEGER',
        'season': 'TEXT',
        'hour': 'INTEGER',
        'solar_kp_index': 'REAL',
        'solar_ap_index': 'INTEGER',  
        'temperature_f': 'REAL', 
        'dew_point': 'REAL', 
        'cloud_cover': 'REAL',
        'precip_in': 'REAL',
        'weather_conditions': 'TEXT'
    }
)
connection.execute("""
    CREATE INDEX IF NOT EXISTS idx_weather_uap_id 
    ON us_uap_2011_weather(uap_id)
""")


# kp index table from processed/ folder
kp_index_df = pd.read_csv("../data/processed/kp_index.csv")

kp_index_df.to_sql(
    name='kp_index', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'datetime': 'TEXT',
        'year': 'INTEGER', 
        'month': 'INTEGER', 
        'day': 'INTEGER', 
        'hour_start': 'REAL', 
        'hour_end': 'REAL', 
        'decimal_day_start': 'REAL',
        'decimal_day_end': 'REAL', 
        'kp': 'REAL', 
        'ap': 'INTEGER', 
        'flag': 'INTEGER'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_datetime ON kp_index(datetime)")

print("\nAll tables created successfully!")

### Display Tables in the SQLite DB (Query 0 = tables_query)

#### Check to ensure all tables are in the relational datase

In [ ]:
tables_query = """
SELECT name FROM sqlite_master 
WHERE type='table' 
AND name NOT LIKE 'sqlite_%';
"""
tables_result = pd.read_sql(tables_query, connection)
tables_result

### KP Index Distribution Comparison (Query 1 = kp_index_pct_query)

**Goal**: Compare the natural distribution of the KP index against the KP index observed at the time of Bigfoot and UAP reports.

#### Key Questions
- How frequently does each KP index level occur in general (background distribution)?
- Do Bigfoot and UAP reports occur during KP index levels at rates significantly different from the background?
- High KP values are known to be rare — do sightings correlate with them more than expected?

#### Output Table Shows
- `kp_index` — Rounded KP index value
- `kp_pct` — Percentage of all KP readings at this level (background)
- `bigfoot_kp_pct` — Percentage of Bigfoot reports at this KP level
- `uap_kp_pct` — Percentage of UAP reports at this KP level
- `bigfoot_kp_diff` — Difference from background (positive = over-represented)
- `uap_kp_diff` — Difference from background (positive = over-represented)

**Interpretation**: 
- Positive differences suggest Bigfoot or UAP reports are more common closer to KP level 0 than expected under normal conditions.
- Otherwise, there is little difference in the distribution.

In [ ]:
kp_index_pct_query = """
WITH
    kp_stats AS (
        SELECT 
            ROUND(k.kp) AS rounded_kp,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS kp_percent
        FROM kp_index k
        GROUP BY ROUND(k.kp)
    ),
    bigfoot_kp AS (
        SELECT 
            ROUND(solar_kp_index) AS rounded_kp,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS bf_kp_percent
        FROM bigfoot_reports
        GROUP BY ROUND(solar_kp_index)
    ),
    uap_kp AS (
        SELECT 
            ROUND(solar_kp_index) AS rounded_kp,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS uap_kp_percent
        FROM uap_reports
        GROUP BY ROUND(solar_kp_index)
    )

SELECT
    k.rounded_kp AS kp_index,
    k.kp_percent AS kp_pct,
    b.bf_kp_percent AS bigfoot_kp_pct,
    u.uap_kp_percent AS uap_kp_pct,
    ROUND(b.bf_kp_percent - k.kp_percent, 2) AS bigfoot_kp_diff,
    ROUND(u.uap_kp_percent - k.kp_percent, 2) AS uap_kp_diff
FROM kp_stats k
LEFT JOIN bigfoot_kp b   ON k.rounded_kp = b.rounded_kp
LEFT JOIN uap_kp u       ON k.rounded_kp = u.rounded_kp
ORDER BY k.rounded_kp ASC;
"""


kp_index_pct_result = pd.read_sql(kp_index_pct_query, connection)
kp_index_pct_result


### Month and Season Comparisson for Bigfoot and UAP sightings
- Is there a Season for Bigfoot or UAP Sightings? 
- How much overlap exists? 

#### Percent Of Each Sighting By Month (Query 2 = month_pct_query)

In [ ]:
month_pct_query = """
WITH
    bigfoot_months AS (
        SELECT 
            month,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS bf_month_percent
        FROM bigfoot_reports
        GROUP BY month
    ),
    uap_months AS (
        SELECT 
            month,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS uap_month_percent
        FROM uap_reports
        GROUP BY month
    )

SELECT
    b.month AS month,
    b.bf_month_percent AS bigfoot_month_pct,
    u.uap_month_percent AS uap_month_pct
FROM bigfoot_months b 
LEFT JOIN uap_months u ON b.month = u.month
ORDER BY b.month ASC;
"""


month_pct_result = pd.read_sql(month_pct_query, connection)
month_pct_result

#### Monthly Bigfoot Reports and UAP Sightings as % of Total

In [ ]:
month_pct_result_long_df = month_pct_result.copy().melt(
    id_vars='month',
    value_vars=['bigfoot_month_pct', 'uap_month_pct'],
    var_name='Type',
    value_name='Percentage'
)

month_pct_result_long_df['Type'] = month_pct_result_long_df['Type'].replace({
    'bigfoot_month_pct': 'Bigfoot',
    'uap_month_pct': 'UAP'
})


hue_order = ['Bigfoot', 'UAP']

stellar_night   = "#280059"   
space_lazer_green = "#10C000"   

plt.figure(figsize=(12, 6.5))
ax = sns.barplot(
    data=month_pct_result_long_df,
    x='month',
    y='Percentage',
    hue='Type',
    hue_order=hue_order,
    palette=[space_lazer_green, stellar_night],
    errorbar=None,
    width=0.8          
)

# Clean spines + subtle grid
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.3)
ax.set_axisbelow(True)

# Annotation 
ax.text( 2.5, 11,
    "All reports appear elevated from June - November →",
    fontsize=11,
    color=stellar_night,
    ha='center',
    fontweight='medium'
)

# Bar labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8.5, rotation=45)


plt.title('Monthly Bigfoot Reports and UAP Sightings as Percent of Total', 
          fontsize=14, pad=20, fontweight='semibold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Percentage of Observations / Reports', fontsize=12)
plt.xticks(rotation=0)          
plt.legend(title='', frameon=False, loc='upper right')

plt.tight_layout()

# write file to ""../plots/"
plot_filename = "../plots/monthly_bar.png"
if os.path.exists(plot_filename):
    print(f"{plot_filename} already exists and will be overwritten.")
plt.savefig(plot_filename, dpi=300, bbox_inches="tight")

plt.show()

### State-Level Bigfoot, UAP & Proximity Report Rates (Query 3 = states_query)

**Goal**: Compare how frequently Bigfoot, UAP, and proximity events are reported across U.S. states, normalized by population.

#### What the query shows:
- Raw counts of reports per state
- Reports **per million people** (rate) for fair comparison between large and small states

#### Columns
- `state_code` — Two-letter state abbreviation
- `population` — 2010 state population
- `bigfoot_total` — Number of Bigfoot reports
- `uap_total` — Number of UAP reports  
- `proximity_total` — Number of proximity events
- `bigfoot_per_mil` — Bigfoot reports per million residents
- `uap_per_mil` — UAP reports per million residents
- `proximity_per_mil` — Proximity events per million residents

**Higher "per million" values = more reports relative to population size.**

In [ ]:
states_query = """
WITH 
    states_pop AS (
        SELECT 
            state_code,
            "2010_population" as population
        FROM states
    ),
    bigfoot AS (
        SELECT
            state_code,
            COUNT(*) as bigfoot_total
        FROM bigfoot_reports
        GROUP BY state_code    
    ),
    uap AS (
        SELECT
            state_code,
            COUNT(*) as uap_total
        FROM uap_reports
        GROUP BY state_code    
    ),
    prox AS (
        SELECT
            state_code, 
            COUNT(*) as proximity_total
        FROM proximity
        GROUP BY state_code
    )
SELECT 
    s.state_code,
    s.population,
    COALESCE(b.bigfoot_total, 0) AS bigfoot_total,
    COALESCE(u.uap_total, 0) AS uap_total,
    COALESCE(p.proximity_total, 0) AS proximity_total,   
    ROUND(
        COALESCE(b.bigfoot_total, 0) * 1000000.0 / s.population, 
        2
    ) AS bigfoot_per_mil,
    ROUND(
        COALESCE(u.uap_total, 0) * 1000000.0 / s.population, 
        2
    ) AS uap_per_mil,
    ROUND(
        COALESCE(p.proximity_total, 0) * 1000000.0 / s.population, 
        2
    ) AS proximity_per_mil
FROM states_pop s
LEFT JOIN bigfoot b ON s.state_code = b.state_code
LEFT JOIN uap u ON s.state_code = u.state_code
LEFT JOIN prox p ON s.state_code = p.state_code
ORDER BY s.state_code; """

states_result = pd.read_sql(states_query, connection)
states_result

### UAP Per Million US States MAP 
- coming soon / in development

### bigfoot Per Million US States MAP
- coming soon / in development

### Proximity Per Million US States MAP 
- coming soon / in development 

#### Sighting Temperature Query (Query 4 = sighting_temp_query)


In [ ]:
sighting_temp_query = """

SELECT
    'Bigfoot' as report_type, 
    bf_id as report_id,
    full_date as date,
    temperature_f
FROM
    bigfoot_weather

UNION ALL

SELECT
    'UAP' as report_type, 
    w.uap_id as report_id,
    u.full_date as date,
    w.temperature_f as temperature_f 
FROM
    us_uap_2011_weather w
INNER JOIN
    uap_reports u 
ON 
    w.uap_id = u.uap_id

"""

sighting_temp_result = pd.read_sql(sighting_temp_query, connection)
sighting_temp_result

### Visualization: Temperatures of UAP and Bigfoot Sightings - Histogram

In [ ]:

uap_temperature = pd.to_numeric(
    sighting_temp_result[sighting_temp_result['report_type'] == 'UAP']['temperature_f']
    .astype(str).str.replace(r".*/.*", "", regex=True)
)
mean_uap_temperature = uap_temperature.mean().round(2)

bigfoot_temperature = pd.to_numeric(
    sighting_temp_result[sighting_temp_result['report_type'] == 'Bigfoot']['temperature_f']
    .astype(str).str.replace(r".*/.*", "", regex=True)
)
mean_bigfoot_temperature = bigfoot_temperature.mean().round(2)


stellar_night = "#280059"
space_lazer_green = "#10C000"

plt.figure(figsize=(10,7))

plt.hist(uap_temperature, bins=40, alpha=0.9, label='UAP', color=stellar_night)

plt.hist(bigfoot_temperature, bins=40, alpha=0.9, label='Bigfoot', color=space_lazer_green)


plt.xlabel("Temperature in °F")
plt.ylabel("Number of Sightings")
plt.title("Bigfoot Likes Cooler Weather", fontsize=18, pad=20)

plt.grid(True, alpha=0.1)

plt.legend(loc="upper left")

ax = plt.gca()

ax.set_ylim(0, 600)

ax.axvline(mean_uap_temperature, linewidth=1, color=stellar_night)
ax.text(mean_uap_temperature + 1.5, 500,
        f"← Average UAP Temperature\n   {mean_uap_temperature}°F",
        color=stellar_night)

ax.axvline(mean_bigfoot_temperature, linewidth=1, color="#056D00")
ax.text(mean_bigfoot_temperature - 46, 500,
        f"Average Bigfoot Temperature →   \n{mean_bigfoot_temperature}°F",
        color="#14291c")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# write file to ""../plots/"

plot_filename = "../plots/bigfoot_uap_temperatures_histogram.png"

if os.path.exists(plot_filename):
    print(f"{plot_filename} already exists and will be overwritten.")
plt.savefig(plot_filename, dpi=300, bbox_inches="tight")


plt.show()


### Cloud Cover % Distribution of Bigfoot and UAP REports (Query 5 = cloud_cover_query)

In [ ]:
# Compare Cloud Cover Of UAP and Bigfoot Sightings
# Since There Are Fewer Bigfoot Reports, We Will Use Percentages 

# Bigfoot percentages by 10-point cloud-cover bins
# UAP percentages by the same bins
# Select with both CTEs side-by-side

cloud_cover_query = """

WITH 


bigfoot_bins AS (
    SELECT
        CASE
            WHEN cloud_cover >= 0  AND cloud_cover < 10  THEN '0-10'
            WHEN cloud_cover >= 10 AND cloud_cover < 20  THEN '10-20'
            WHEN cloud_cover >= 20 AND cloud_cover < 30  THEN '20-30'
            WHEN cloud_cover >= 30 AND cloud_cover < 40  THEN '30-40'
            WHEN cloud_cover >= 40 AND cloud_cover < 50  THEN '40-50'
            WHEN cloud_cover >= 50 AND cloud_cover < 60  THEN '50-60'
            WHEN cloud_cover >= 60 AND cloud_cover < 70  THEN '60-70'
            WHEN cloud_cover >= 70 AND cloud_cover < 80  THEN '70-80'
            WHEN cloud_cover >= 80 AND cloud_cover < 90  THEN '80-90'
            WHEN cloud_cover >= 90 AND cloud_cover <= 100 THEN '90-100'
            ELSE 'out-of-range'
        END AS cloud_range,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS bigfoot_pct
    FROM bigfoot_weather
    WHERE cloud_cover BETWEEN 0 AND 100          -- safety filter
    GROUP BY 1
),


uap_bins AS (
    SELECT
        CASE
            WHEN w.cloud_cover >= 0  AND w.cloud_cover < 10  THEN '0-10'
            WHEN w.cloud_cover >= 10 AND w.cloud_cover < 20  THEN '10-20'
            WHEN w.cloud_cover >= 20 AND w.cloud_cover < 30  THEN '20-30'
            WHEN w.cloud_cover >= 30 AND w.cloud_cover < 40  THEN '30-40'
            WHEN w.cloud_cover >= 40 AND w.cloud_cover < 50  THEN '40-50'
            WHEN w.cloud_cover >= 50 AND w.cloud_cover < 60  THEN '50-60'
            WHEN w.cloud_cover >= 60 AND w.cloud_cover < 70  THEN '60-70'
            WHEN w.cloud_cover >= 70 AND w.cloud_cover < 80  THEN '70-80'
            WHEN w.cloud_cover >= 80 AND w.cloud_cover < 90  THEN '80-90'
            WHEN w.cloud_cover >= 90 AND w.cloud_cover <= 100 THEN '90-100'
            ELSE 'out-of-range'
        END AS cloud_range,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS uap_pct
    FROM us_uap_2011_weather w
    INNER JOIN uap_reports u ON w.uap_id = u.uap_id
    WHERE w.cloud_cover BETWEEN 0 AND 100
    GROUP BY 1
)


SELECT
    COALESCE(b.cloud_range, u.cloud_range) AS cloud_range,
    COALESCE(b.bigfoot_pct, 0)             AS bigfoot_pct,
    COALESCE(u.uap_pct, 0)                 AS uap_pct
FROM bigfoot_bins b
FULL OUTER JOIN uap_bins u 
    ON b.cloud_range = u.cloud_range
ORDER BY 
    CASE COALESCE(b.cloud_range, u.cloud_range)
        WHEN '0-10'   THEN 1
        WHEN '10-20'  THEN 2
        WHEN '20-30'  THEN 3
        WHEN '30-40'  THEN 4
        WHEN '40-50'  THEN 5
        WHEN '50-60'  THEN 6
        WHEN '60-70'  THEN 7
        WHEN '70-80'  THEN 8
        WHEN '80-90'  THEN 9
        WHEN '90-100' THEN 10
        ELSE 11
    END;


"""

cloud_cover_result = pd.read_sql(cloud_cover_query, connection)
cloud_cover_result



### Visualization: Cloud Cover % Distribution of UAP and Bigfoot Reports - Bar Chart

In [ ]:
# Compare Cloud Cover of UAP and Bigfoot Sightings

# Melt First

cloud_cover_long_df = cloud_cover_result.melt(
    id_vars='cloud_range',
    value_vars=['bigfoot_pct', 'uap_pct'],
    var_name='Type',
    value_name='Percentage'  
)


cloud_cover_long_df['Type'] = cloud_cover_long_df['Type'].replace({
    'bigfoot_pct': 'Bigfoot',
    'uap_pct': 'UAP'
})

# SNS Grouped Barplot 

stellar_night = "#280059"
space_lazer_green = "#10C000"

plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=cloud_cover_long_df,
    x='cloud_range',
    y='Percentage',
    hue='Type',
    palette=[space_lazer_green, stellar_night],  
    errorbar=None
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.text( 0.5, 30, f"← UAPs Prefer Clear Skies", color=stellar_night)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

plt.title('Cloud Cover Distribution: Bigfoot vs UAP Reports', fontsize=14, pad=15)
plt.xlabel('Cloud Cover Range (%)', fontsize=12)
plt.ylabel('Percentage of Reports', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='')
plt.tight_layout()

# write file to ""../plots/"

plot_filename = "../plots/cloud_cover_distribution_all_reports_bar.png"

if os.path.exists(plot_filename):
    print(f"{plot_filename} already exists and will be overwritten.")
plt.savefig(plot_filename, dpi=300, bbox_inches="tight")


plt.show()



### Visualization: Historical Kp Index Distribution VS Kp Index Distribution of UAP and Bigfoot Reports - Bar Chart

In [ ]:
# Historical KP vs Bigfoot & UAP Report Conditions

kp_result_long_df = kp_index_pct_result.copy().melt(
    id_vars='kp_index',
    value_vars=['kp_pct', 'bigfoot_kp_pct', 'uap_kp_pct'],
    var_name='Type',
    value_name='Percentage'
)

kp_result_long_df['Type'] = kp_result_long_df['Type'].replace({
    'kp_pct': 'Historical KP',
    'bigfoot_kp_pct': 'Bigfoot',
    'uap_kp_pct': 'UAP'
})


hue_order = ['Historical KP', 'Bigfoot', 'UAP']

stellar_night   = "#280059"   
space_lazer_green = "#10C000" 
solar_flare     = "#8a009c"   

plt.figure(figsize=(12, 6.5))
ax = sns.barplot(
    data=kp_result_long_df,
    x='kp_index',
    y='Percentage',
    hue='Type',
    hue_order=hue_order,
    palette=[solar_flare, space_lazer_green, stellar_night],
    errorbar=None,
    width=0.8          
)

# Clean spines + subtle grid
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.3)
ax.set_axisbelow(True)

# Annotation 
ax.text( 5, 20,
    "Bigfoot & UAPs appear elevated \nat quiet KP (0–1)",
    fontsize=11,
    color=stellar_night,
    ha='center',
    fontweight='medium'
)

# Bar labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8.5, rotation=45)


plt.title('Geomagnetic Activity (Rounded KP Index):\nHistorical Baseline vs. Bigfoot & UAP Reports',
          fontsize=14, pad=18, fontweight='semibold')
plt.xlabel('Rounded KP Index', fontsize=12)
plt.ylabel('Percentage of Observations / Reports', fontsize=12)
plt.xticks(rotation=0)          
plt.legend(title='', frameon=False, loc='upper right')

plt.tight_layout()

# write file to ""../plots/"
plot_filename = "../plots/kp_index_distribution_all_reports_bar_v2.png"
if os.path.exists(plot_filename):
    print(f"{plot_filename} already exists and will be overwritten.")
plt.savefig(plot_filename, dpi=300, bbox_inches="tight")

plt.show()

### UAP Locations (Query 6 = uap_us_48_location_query)

In [ ]:
uap_us_48_location_query = """
SELECT 
    u.uap_id,
    u.datetime,
    u.state_code as state_code,
    s."2010_population" as state_population,
    s.pop_density as pop_density, 
    u.latitude as latitude,
    u.longitude as longitude
FROM uap_reports u
LEFT JOIN states s
ON u.state_code = s.state_code

WHERE u.state_code NOT IN ('AK', 'HI', 'PR')
;
"""

uap_us_48_location_result_df = pd.read_sql(uap_us_48_location_query, connection)
uap_us_48_location_result_df


### Visualization: UAP Reports Scatter Plot US Map 
- Marker Size determined by State population density 

In [ ]:
# State Population Density Weighted Plot – Contiguous 48 States

max_pop = uap_us_48_location_result_df['pop_density'].max()
uap_us_48_location_result_df["population_factor"] = (
    (max_pop * 1.5) / (uap_us_48_location_result_df['pop_density'] * 6)
) 

uap_us_48_location_gdf = gpd.GeoDataFrame(
    uap_us_48_location_result_df,
    geometry=gpd.points_from_xy(
        uap_us_48_location_result_df.longitude,
        uap_us_48_location_result_df.latitude
    ),
    crs="EPSG:4326"
)

url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_1_states_provinces.zip"
states = gpd.read_file(url)

usa = states[states["admin"] == "United States of America"]
usa = usa[~usa["name"].isin(["Alaska", "Hawaii"])]

fig, ax = plt.subplots(figsize=(13, 8))

usa.plot(ax=ax, color="white", edgecolor="black", linewidth=0.6)
uap_us_48_location_gdf.plot(
    ax=ax,
    color=stellar_night,
    markersize=uap_us_48_location_gdf['population_factor'],
    alpha=0.5,              
    edgecolor='none'
)

# Title + subtitle
ax.set_title("Contiguous US UAP Sightings", fontsize=18, fontweight='semibold', pad=14)
ax.text(
    0.5, 0.98,
    "* Marker size is inversely proportional to state population density",
    transform=ax.transAxes,
    ha='center',
    fontsize=12,
    style='italic',
    color="#323232"
)

ax.set_axis_off()
plt.tight_layout()

# write file to ""../plots/"
plot_filename = "../plots/contigous_us_uap_plot_map.png"
if os.path.exists(plot_filename):
    print(f"{plot_filename} already exists and will be overwritten.")
plt.savefig(plot_filename, dpi=300, bbox_inches="tight")

plt.show()



### US Bigfoot Locations (Query 7 = bigfoot_us_48_location_query)

In [ ]:
bigfoot_us_48_location_query = """
SELECT 
    b.bf_id,
    b.full_date,
    b.state_code as state_code,
    s."2010_population" as state_population, 
    b.latitude as latitude,
    b.longitude as longitude
FROM bigfoot_reports b
LEFT JOIN states s
ON b.state_code = s.state_code

WHERE 
    b.state_code NOT IN ('AK', 'HI', 'PR')
    AND
    b.longitude > -125
;
"""

bigfoot_us_48_location_result_df = pd.read_sql(bigfoot_us_48_location_query, connection)
bigfoot_us_48_location_result_df.head()

### Visualization: All Bigfoot Reports Scatter Plot US Map

In [ ]:
# US Bigfoot Sightings

bigfoot_us_48_location_gdf = gpd.GeoDataFrame(
    bigfoot_us_48_location_result_df,
    geometry=gpd.points_from_xy(
        bigfoot_us_48_location_result_df.longitude,
        bigfoot_us_48_location_result_df.latitude
    ),
    crs="EPSG:4326"
)

url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_1_states_provinces.zip"
states = gpd.read_file(url)

usa = states[states["admin"] == "United States of America"]
usa = usa[~usa["name"].isin(["Alaska", "Hawaii"])]

fig, ax = plt.subplots(figsize=(13, 8))

usa.plot(ax=ax, color="white", edgecolor="black", linewidth=0.6)
bigfoot_us_48_location_gdf.plot(
    ax=ax,
    color=space_lazer_green,
    markersize=18,
    alpha=0.65,              
    edgecolor='none'
)

# Title + subtitle
ax.set_title("Contiguous US Bigfoot Reports", fontsize=18, fontweight='semibold', pad=16)


ax.set_axis_off()
plt.tight_layout()

# write file to ""../plots/"

plot_filename = "../plots/contiguous_us_bigfoot_reports.png"
if os.path.exists(plot_filename):
    print(f"{plot_filename} already exists and will be overwritten.")
plt.savefig(plot_filename, dpi=300, bbox_inches="tight")

plt.show()



### Proximity Scotter Plot Map (Query 8 = proximity_us_48_location_query)

In [ ]:
# Query for Proximity Locations

proximity_us_48_location_query = """
SELECT 
    p.uap_id,
    p.full_date_uap full_date_uap,
    p.state_code as state_code,
    s."2010_population" as state_population, 
    p.latitude_uap as latitude,
    p.longitude_uap as longitude,
    p.proximity_score as proximity_score
FROM proximity p
LEFT JOIN states s
ON p.state_code = s.state_code

WHERE 
    p.state_code NOT IN ('AK', 'HI', 'PR')
    AND
    p.longitude_uap > -125
;
"""

proximity_us_48_location_result_df = pd.read_sql(proximity_us_48_location_query, connection)
proximity_us_48_location_result_df.head()

### Visualization: Proximity Score Weighted Scatter Plot US Map

In [ ]:
# Proximity Score Weighted Plot – Contiguous 48 States

proximity_us_48_location_result_df["proximity_factor"] = proximity_us_48_location_result_df['proximity_score'] * 10


proximity_us_48_location_gdf = gpd.GeoDataFrame(
    proximity_us_48_location_result_df,
    geometry=gpd.points_from_xy(
        proximity_us_48_location_result_df.longitude,
        proximity_us_48_location_result_df.latitude
    ),
    crs="EPSG:4326"
)

url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_1_states_provinces.zip"
states = gpd.read_file(url)

usa = states[states["admin"] == "United States of America"]
usa = usa[~usa["name"].isin(["Alaska", "Hawaii"])]

fig, ax = plt.subplots(figsize=(13, 8))

usa.plot(ax=ax, color="white", edgecolor="black", linewidth=0.6)
proximity_us_48_location_gdf.plot(
    ax=ax,
    color=stellar_night,
    markersize=proximity_us_48_location_gdf['proximity_factor'],
    alpha=0.65,              
    edgecolor='none'
)

# Title + subtitle
ax.set_title("Contiguous US UAP Bigfoot Close Proximity", fontsize=18, fontweight='semibold', pad=14)
ax.text(
    0.5, 0.98,
    "* Marker size is proportional to Proximity Score",
    transform=ax.transAxes,
    ha='center',
    fontsize=12,
    style='italic',
    color="#323232"
)

ax.set_axis_off()
plt.tight_layout()

plot_filename = "../plots/proximity_map.png"
if os.path.exists(plot_filename):
    print(f"{plot_filename} already exists and will be overwritten.")
plt.savefig(plot_filename, dpi=300, bbox_inches="tight")

plt.show()

